In [23]:
import pandas as pd
import requests
import statsmodels as sm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as plt
import seaborn as sns
import csv
import time
from scipy import io
import io
from scipy import stats
import statsmodels.api as sm
import importlib

In questo notebook ho provato ad ottenere i dati sui veicoli coinvolti in incidenti tramite richiesta API sul sito [ESPLORA DATI - ISTAT](https://esploradati.istat.it/)
=====================================================================================================================================


In [ ]:
lista_territori = [
    "ITC1", "ITC2", "ITC3", "ITC4", "ITD1", 
    "ITD2", "ITD3", "ITD4", "ITD5", "ITE1", 
    "ITE2", "ITE3", "ITE4", "ITF1", "ITF2", 
    "ITF3", "ITF4", "ITF5", "ITF6", "ITG1", "ITG2"
]

chunk_size = 5
regioni_scaricate = []

for i in range(0, len(lista_territori), chunk_size):
    chunk = lista_territori[i : i + chunk_size]
    territori_str = "+".join(chunk)

    url = f"https://esploradati.istat.it/SDMXWS/rest/data/IT1,41_271_DF_DCIS_VEICOLIINCID1_1,1.0/A.{territori_str}.VEHICINV.9.1+2+3+4+5+6+7+9.1+2+3+9.0+1+2+3+4+5+6+7+8+9+99.99/ALL/?detail=full&startPeriod=2024-01-01&endPeriod=2024-12-31&dimensionAtObservation=TIME_PERIOD"

    try:
        headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
        response = requests.get(url, headers=headers, timeout=30)

        if response.status_code == 200:
            df_temp = pd.read_csv(io.StringIO(response.text))
            regioni_scaricate.append(df_temp)
            print(f"Territorio {territori_str} scaricato con successo.")
        elif response.status_code == 429:
            break
        else:
            print(f"Errore {response.status_code} per il blocco {i // chunk_size + 1}")

    except requests.exceptions.Timeout:
        print(f"Timeout per il blocco con territori: {territori_str}. Riprovare riducendo il chunk_size.")
    except Exception as e:
        print(f"Errore di connessione: {e}")

    print("Attesa")
    time.sleep(18)

# Unione finale nel DataFrame chiamato 'veicoli_coinvolti'
if regioni_scaricate:
    veicoli_coinvolti = pd.concat(regioni_scaricate, ignore_index=True)
    display(veicoli_coinvolti)
else:
    print("Nessun dato scaricato")


Timeout per il blocco con territori: ITC1+ITC2+ITC3+ITC4+ITD1. Riprovare riducendo il chunk_size.
Attesa
Timeout per il blocco con territori: ITD2+ITD3+ITD4+ITD5+ITE1. Riprovare riducendo il chunk_size.
Attesa
Timeout per il blocco con territori: ITE2+ITE3+ITE4+ITF1+ITF2. Riprovare riducendo il chunk_size.
Attesa
Timeout per il blocco con territori: ITF3+ITF4+ITF5+ITF6+ITG1. Riprovare riducendo il chunk_size.
Attesa
Territorio ITG2 scaricato con successo.
Attesa


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,ACCIDENT_LOCALIZATON,INTERSECTION,TY_ROAD_ACCIDENT,VEHICLE_TYPE,MONTH,TIME_PERIOD,...,NOTE_DATA_TYPE,NOTE_ACCIDENT_LOCALIZATON,NOTE_INTERSECTION,NOTE_TY_ROAD_ACCIDENT,NOTE_VEHICLE_TYPE,NOTE_MONTH,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,1,1,0,99,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,1,1,1,99,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,1,1,1,99,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,1,1,2,99,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,1,1,2,99,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
398,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,9,9,8,99,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
399,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,9,9,9,99,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
400,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,9,9,9,99,2025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
401,IT1:41_271_DF_DCIS_VEICOLIINCID1_1(1.0),A,ITG2,VEHICINV,9,9,9,99,99,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
veicoli_coinvolti.to_csv("./data/veicoli_coinvolti.csv", index=False, sep=';', decimal=',')


**ESTRAZIONE E TRASFORMAZIONE DATI**
------------------------------------

In [64]:
veicoli_coinvolti = pd.read_csv("./data/veicoli_coinvolti.csv", sep=";", decimal=',')
incidenti_agg = pd.read_csv("./data/incidenti_agg.csv", sep=";", decimal=',')

In [65]:
# Controllo quali colonne ho a disposizione nel DataFrame
print(veicoli_coinvolti.columns)
# e controllo i tipi di veicoli
print(veicoli_coinvolti["VEHICLE_TYPE"].unique())

Index(['DATAFLOW', 'FREQ', 'REF_AREA', 'DATA_TYPE', 'ACCIDENT_LOCALIZATON',
       'INTERSECTION', 'TY_ROAD_ACCIDENT', 'VEHICLE_TYPE', 'MONTH',
       'TIME_PERIOD', 'OBS_VALUE', 'OBS_STATUS', 'NOTE_DS', 'NOTE_REF_AREA',
       'NOTE_DATA_TYPE', 'NOTE_ACCIDENT_LOCALIZATON', 'NOTE_INTERSECTION',
       'NOTE_TY_ROAD_ACCIDENT', 'NOTE_VEHICLE_TYPE', 'NOTE_MONTH',
       'NOTE_TIME_PERIOD', 'BASE_PER', 'UNIT_MEAS', 'UNIT_MULT'],
      dtype='str')
[ 0  1  2  3  4  5  6  7  8  9 99]


**CREAZIONE PIVOT**
-------------------

In [73]:
veicoli_coinvolti = veicoli_coinvolti[veicoli_coinvolti["TIME_PERIOD"] == 2024]

pivot_veicoli_2024 = pd.pivot_table(
    veicoli_coinvolti,
    index=["REF_AREA"],
    columns="VEHICLE_TYPE",
    values="OBS_VALUE",
    aggfunc="sum",
)
pivot_veicoli_2024 = pivot_veicoli_2024.reset_index()
pivot_veicoli_2024.head()

VEHICLE_TYPE,REF_AREA,0,1,2,3,4,5,6,7,8,9,99
0,ITC1,108.0,55636.0,696.0,152.0,5012.0,5400.0,1240.0,6728.0,20.0,1908.0,76900.0
1,ITC2,NaN,1448.0,28.0,NaN,160.0,84.0,32.0,200.0,NaN,36.0,1988.0
2,ITC3,36.0,25788.0,1892.0,NaN,3984.0,1848.0,1484.0,18640.0,8.0,1224.0,54904.0
3,ITC4,272.0,139392.0,1184.0,224.0,14780.0,21808.0,5736.0,30456.0,32.0,6056.0,219940.0
4,ITD1,12.0,6976.0,264.0,NaN,1008.0,1712.0,188.0,1960.0,4.0,300.0,12424.0


**SOSTITUZIONE CODICI REGIONALI**
---------------------------------

In [74]:
CODICI_REGIONI = {
    "ITC1": "Piemonte",
    "ITC2": "Valle d'Aosta",
    "ITC3": "Liguria",
    "ITC4": "Lombardia",
    "ITD1": "P.A. Bolzano",
    "ITD2": "P.A. Trento",
    "ITD3": "Veneto",
    "ITD4": "Friuli-Venezia Giulia",
    "ITD5": "Emilia-Romagna",
    "ITE1": "Toscana",
    "ITE2": "Umbria",
    "ITE3": "Marche",
    "ITE4": "Lazio",
    "ITF1": "Abruzzo",
    "ITF2": "Molise",
    "ITF3": "Campania",
    "ITF4": "Puglia",
    "ITF5": "Basilicata",
    "ITF6": "Calabria",
    "ITG1": "Sicilia",
    "ITG2": "Sardegna",
}

# 2. Troviamo la colonna dei codici (dovrebbe essere la prima) e sostituiamo i codici con i nomi
colonna_codici = pivot_veicoli_2024.columns[0]
pivot_veicoli_2024[colonna_codici] = pivot_veicoli_2024[colonna_codici].replace(
    CODICI_REGIONI
)

# 3. La rinominiamo definitivamente in CODICE_REGIONE
pivot_veicoli_2024 = pivot_veicoli_2024.rename(
    columns={colonna_codici: "CODICE_REGIONE"}
)

pivot_veicoli_2024.columns.name = None

# Mostriamo il risultato finale
display(pivot_veicoli_2024)

,CODICE_REGIONE,0,1,2,3,4,5,6,7,8,9,99
0,Piemonte,108.0,55636.0,696.0,152.0,5012.0,5400.0,1240.0,6728.0,20.0,1908.0,76900.0
1,Valle d'Aosta,NaN,1448.0,28.0,NaN,160.0,84.0,32.0,200.0,NaN,36.0,1988.0
2,Liguria,36.0,25788.0,1892.0,NaN,3984.0,1848.0,1484.0,18640.0,8.0,1224.0,54904.0
3,Lombardia,272.0,139392.0,1184.0,224.0,14780.0,21808.0,5736.0,30456.0,32.0,6056.0,219940.0
4,P.A. Bolzano,12.0,6976.0,264.0,NaN,1008.0,1712.0,188.0,1960.0,4.0,300.0,12424.0
5,P.A. Trento,4.0,5784.0,72.0,NaN,748.0,760.0,208.0,1448.0,12.0,228.0,9264.0
6,Veneto,104.0,61536.0,552.0,24.0,6956.0,10412.0,2360.0,10580.0,12.0,1724.0,94260.0
7,Friuli-Venezia Giulia,28.0,15504.0,168.0,12.0,1464.0,1532.0,652.0,2988.0,16.0,480.0,22844.0
8,Emilia-Romagna,152.0,77116.0,732.0,NaN,9336.0,14960.0,2908.0,13208.0,12.0,3340.0,121764.0
9,Toscana,192.0,68272.0,816.0,12.0,7008.0,7984.0,5112.0,19260.0,92.0,2268.0,111016.0


**SOSTITUZIONE CODICI DEI VEICOLI**
-----------------------------------

In [77]:
CODICI_VEICOLI = {
    0: "Quadricicli",
    1: "Autovetture",
    2: "Autobus",
    3: "Tram",
    4: "Autocarri",
    5: "Velocipede",
    6: "Ciclomotori",
    7: "Motocicli",
    8: "Motocarri",
    9: "Altro",
    99: "Totale",
}
pivot_veicoli_2024 = pivot_veicoli_2024.rename(columns=CODICI_VEICOLI)


pivot_veicoli_2024

,CODICE_REGIONE,Quadricicli,Autovetture,Autobus,Tram,Autocarri,Velocipede,Ciclomotori,Motocicli,Motocarri,Altro,Totale
3,Lombardia,272.0,139392.0,1184.0,224.0,14780.0,21808.0,5736.0,30456.0,32.0,6056.0,219940.0
12,Lazio,676.0,105452.0,1288.0,28.0,8040.0,5668.0,2728.0,31636.0,16.0,4484.0,160016.0
8,Emilia-Romagna,152.0,77116.0,732.0,0.0,9336.0,14960.0,2908.0,13208.0,12.0,3340.0,121764.0
9,Toscana,192.0,68272.0,816.0,12.0,7008.0,7984.0,5112.0,19260.0,92.0,2268.0,111016.0
6,Veneto,104.0,61536.0,552.0,24.0,6956.0,10412.0,2360.0,10580.0,12.0,1724.0,94260.0
19,Sicilia,476.0,54976.0,292.0,24.0,3356.0,2420.0,3476.0,16992.0,96.0,1636.0,83744.0
16,Puglia,60.0,60844.0,368.0,0.0,3616.0,3104.0,1824.0,10176.0,112.0,2484.0,82588.0
15,Campania,200.0,52760.0,368.0,8.0,3308.0,2208.0,1584.0,18420.0,44.0,1876.0,80776.0
0,Piemonte,108.0,55636.0,696.0,152.0,5012.0,5400.0,1240.0,6728.0,20.0,1908.0,76900.0
2,Liguria,36.0,25788.0,1892.0,0.0,3984.0,1848.0,1484.0,18640.0,8.0,1224.0,54904.0


**SOSTITUZIONE NaN**
--------------------

In [78]:
pivot_veicoli_2024 = pivot_veicoli_2024.fillna(0)
pivot_veicoli_2024 = pivot_veicoli_2024.sort_values(by="Totale", ascending=False)
pivot_veicoli_2024

,CODICE_REGIONE,Quadricicli,Autovetture,Autobus,Tram,Autocarri,Velocipede,Ciclomotori,Motocicli,Motocarri,Altro,Totale
3,Lombardia,272.0,139392.0,1184.0,224.0,14780.0,21808.0,5736.0,30456.0,32.0,6056.0,219940.0
12,Lazio,676.0,105452.0,1288.0,28.0,8040.0,5668.0,2728.0,31636.0,16.0,4484.0,160016.0
8,Emilia-Romagna,152.0,77116.0,732.0,0.0,9336.0,14960.0,2908.0,13208.0,12.0,3340.0,121764.0
9,Toscana,192.0,68272.0,816.0,12.0,7008.0,7984.0,5112.0,19260.0,92.0,2268.0,111016.0
6,Veneto,104.0,61536.0,552.0,24.0,6956.0,10412.0,2360.0,10580.0,12.0,1724.0,94260.0
19,Sicilia,476.0,54976.0,292.0,24.0,3356.0,2420.0,3476.0,16992.0,96.0,1636.0,83744.0
16,Puglia,60.0,60844.0,368.0,0.0,3616.0,3104.0,1824.0,10176.0,112.0,2484.0,82588.0
15,Campania,200.0,52760.0,368.0,8.0,3308.0,2208.0,1584.0,18420.0,44.0,1876.0,80776.0
0,Piemonte,108.0,55636.0,696.0,152.0,5012.0,5400.0,1240.0,6728.0,20.0,1908.0,76900.0
2,Liguria,36.0,25788.0,1892.0,0.0,3984.0,1848.0,1484.0,18640.0,8.0,1224.0,54904.0


**esportazione del csv**

In [79]:
pivot_veicoli_2024.to_csv("./data/pivot_veicoli_2024.csv", index=False, sep=';', decimal=',')